# 과제 5,6,7

5. 활성화 함수를 직접 정의하고, 활성화 함수를 적용한 출력을 계산하고, 결과를 그래프로 시각화하세요.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def swiglu(x):
    return x * sigmoid(x) 

x = np.array([-3, -2, -1, 0, 1, 2, 3])

print("Sigmoid:", sigmoid(x))
print("ReLU:", relu(x))
print("Swiglu:", swiglu(x))

plt.plot(x, sigmoid(x), marker="o", label="sigmoid")
plt.plot(x, relu(x), marker="o", label="ReLU")
plt.plot(x, swiglu(x), marker="o", label="SwiGLU")

plt.legend()
plt.grid()
plt.show()

6. 비선형 데이터셋을 생성하고, MLP(다층 퍼셉트론) 모델을 설계하고 학습시켜 분류를 수행하세요.


In [ ]:
from sklearn.datasets import make_moons # 가상 데이터셋 생성 함수인데, 비선형 용도

X,y = make_moons(n_samples=100, noise=0.2, random_state=42)

print("x.shape:", X.shape)
print("y.shape:", y.shape)
print(X[:5]) 
print(y[:5]) 

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

import torch

# NumPy -> Tensor 변환
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

In [ ]:
# SwiGLU 사용을 위한 정의 
import torch.nn as nn

class SwiGLU(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2,16)
        self.fc2 = nn.Linear(16,16)
        self.fc3 = nn.Linear(16,2)
        self.swiglu = SwiGLU()
        
    def forward(self,x):
        x = self.swiglu(self.fc1(x))
        x = self.swiglu(self.fc2(x))
        x = self.fc3(x)
        return x
    
model = MLP()
print(model)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters())

epochs = 100
best_loss = float('inf')
early_stop = 10
counter = 0

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()

    if loss.item() < best_loss:
        best_loss = loss.item()
        counter = 0
    else:
        counter += 1

    if epoch % 10 == 0:
        print(f"Epoch {epochs}, Loss: {loss.item():.4f}")

    if counter >= early_stop:
        print("Epoch", epochs)
        break
        

In [ ]:
model.eval()
with torch.no_grad():
    output = model(X_test)
    pred = torch.argmax(output, dim=1)
    accuracy = (pred == y_test).float().mean()
    print(f"Accuracy: {accuracy:.4f}")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].scatter(
    X_test[:, 0],
    X_test[:, 1],
    c=y_test,
)

ax[0].set_title("Test")

ax[1].scatter(
    X_test[:, 0],
    X_test[:, 1],
    c=pred,
)

ax[1].set_title("pred")

plt.show()

7. CNN(Convolutional Neural Network)을 직접 구성하여 이미지 분류를 수행하세요.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()

X = digits.images   # 이미지 data
y = digits.target   # 정답 라벨

print("X shape:", X.shape)
print("y shape:", y.shape)

print("첫 번째 이미지 라벨:", y[0])
print(X[0])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# (batch_size, channel, height, width) 형태로 변환
X_train = torch.FloatTensor(X_train).unsqueeze(1) # channel 추가
X_test = torch.FloatTensor(X_test).unsqueeze(1)

y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1,out_channels=16,kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2,stride=2)
        self.fc1 = nn.Linear(16 * 4 * 4,64)
        self.fc2 = nn.Linear(64,10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)       
        x = self.relu(x)    
        x = self.pool(x)        
        x = x.view(-1, 16 * 4 * 4)
        x = self.fc1(x)         
        x = self.relu(x)
        x = self.fc2(x)         

        return x

model = CNN()

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

epochs = 100
best_loss = float("inf")
early_stop = 10
counter = 0
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()
    if loss.item() < best_loss:
        best_loss = loss.item()
        counter = 0
    else:
        counter += 1
    if epoch % 10 == 0:
        print(f"Epoch {epoch + 1},Loss: {loss.item():.4f}")

    if counter >= early_stop:
        print(f"Early Stopping at Epoch {epoch + 1}")   
        break
        

In [ ]:
model.eval()
with torch.no_grad():
    output = model(X_test)
    pred = torch.argmax(output, dim=1)
    accuracy = (pred == y_test).float().mean()

print(f"Accuracy: {accuracy:.4f}")

In [ ]:
plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i].squeeze(),cmap="gray")
    plt.title(f"True:{y_test[i]}\nPred:{pred[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()